In [1]:
import os
from sedona.spark import SedonaContext
from sedona.spark import dataframe_to_arrow
from sedona.spark.geoarrow import create_spatial_dataframe
from sedona.spark.maps.SedonaKepler import SedonaKepler

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/25 22:28:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/25 22:28:38 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/11/25 22:28:38 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/11/25 22:28:38 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/11/25 22:28:38 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/11/25 22:28:38 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/11/25 22:28:38 WARN SimpleFunctionRegistry: The function st_envelop

# Load input data

In [3]:
import pyspark.sql.functions as f

paris_places = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/paris_places")

# Create H3 cells for Paris

In [4]:
paris_polygon_wkt = "POLYGON((2.2241 48.8156, 2.4699 48.8156, 2.4699 48.9022, 2.2241 48.9022, 2.2241 48.8156))"

In [5]:
h3_cells = sedona.sql(
    f"""
    WITH h3_cells AS (
        SELECT
            id,
            ST_H3ToGeom(ARRAY(id))[0] AS geom
        LATERAL VIEW EXPLODE(ST_H3CellIDs(ST_GeomFromText('{paris_polygon_wkt}'), 9, true)) AS id
    )
    SELECT 
        id,
        geom,
        ST_X(ST_Centroid(geom)) AS lon,
        ST_Y(ST_Centroid(geom)) AS lat
    FROM h3_cells
    """
)

In [6]:
SedonaKepler.create_map(h3_cells, "h3_cells")

/usr/local/lib/python3.10/dist-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


KeplerGl(data={'h3_cells': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20…

In [7]:
h3_cells.createOrReplaceTempView("h3_cells")

# Filter data to categories

In [8]:
categories = [
    'restaurant',
    'shopping',
    'bakery',
    'education',
    'school',
    'pharmacy', 
    'cafe',
    'theatre',
    'transportation',
    'hospital',
    'park'
]

paris_places.\
    where(f"categories.primary IN {tuple(categories)}").\
    selectExpr("categories.primary AS category", "geometry").\
    createOrReplaceTempView("selected_categories")

In [9]:
sedona.sql("select * from selected_categories").show()

[Stage 5:>                                                          (0 + 1) / 1]

+--------------+--------------------+
|      category|            geometry|
+--------------+--------------------+
|      hospital|POINT (2.2245171 ...|
|      hospital|POINT (2.224481 4...|
|        school|POINT (2.2271923 ...|
|        school|POINT (2.2248187 ...|
|    restaurant|POINT (2.2278196 ...|
|        bakery|POINT (2.2281414 ...|
|    restaurant|POINT (2.2277854 ...|
|        bakery|POINT (2.2280888 ...|
|     education|POINT (2.2377495 ...|
|        school|POINT (2.2385001 ...|
|transportation|POINT (2.2401927 ...|
|    restaurant|POINT (2.24062 48...|
|    restaurant|POINT (2.2473512 ...|
|    restaurant|POINT (2.2476 48....|
|        bakery|POINT (2.2492841 ...|
|       theatre|POINT (2.2517361 ...|
|          park|POINT (2.2425 48....|
|    restaurant|POINT (2.2437786 ...|
|        school|POINT (2.2452141 ...|
|      shopping|POINT (2.2466481 ...|
+--------------+--------------------+
only showing top 20 rows



# Create walk catchments

In [10]:
import requests
from shapely.geometry import shape, MultiPolygon

OPEN_ROUTING_URL = "http://ors:8082"

def walk_time_polygon(lon: float, lat: float, minutes: int):
    body = {
      "locations": [[lon, lat]],
      "range": [minutes * 60],
      "range_type": "time"
    }

    response = requests.post(
        url=f"{OPEN_ROUTING_URL}/ors/v2/isochrones/foot-walking",
        json=body
    )

    if response.status_code != 200:
        return None
        
    response_data = response.json()
    features = response_data["features"]
    shapely_polygons = [
        shape(feature["geometry"]) for feature in features 
        if feature["geometry"]["type"] == 'Polygon'
    ]
    
    return MultiPolygon(shapely_polygons)

In [11]:
import pyspark.sql.functions as f
import sedona.spark.sql.types as st
import shapely.geometry.base as b
 
def create_walk_catchment(
    lon: float,
    lat: float,
    minutes: int
) -> b.BaseGeometry:
    return walk_time_polygon(lon, lat, minutes)
 
create_walk_catchment_udf = f.udf(
    create_walk_catchment,
    st.GeometryType()
)
 
sedona.udf.register(
    "ST_GetWalkCatchment",
    create_walk_catchment_udf
)

In [12]:
catchments_sample = sedona.sql(
"""
SELECT
    id,
    ST_GetWalkCatchment(lon, lat, 10) AS walk_catchment,
    geom
FROM h3_cells
"""
).limit(100)

SedonaKepler.create_map(catchments_sample, "catchments")

KeplerGl(data={'catchments': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, …

In [13]:
sedona.sql(
    """
    WITH catchments AS (
        SELECT
            id,
            ST_GetWalkCatchment(lon, lat, 10) AS walk_catchment,
            geom
        FROM h3_cells
    ),
    joined AS (
        SELECT 
            *,
            c.geom AS grid_geom
        FROM catchments AS c
        JOIN selected_categories AS S ON ST_Intersects(s.geometry, c.walk_catchment)
    )
    SELECT
        id,
        category,
        count(*) AS count,
        FIRST(grid_geom) AS geom
    FROM joined
    GROUP BY id, category
    """
).createOrReplaceTempView("catchments_count")

In [14]:
sedona.sql("SELECT * FROM catchments_count").show(10)

[Stage 8:===============================================>           (4 + 1) / 5]

+------------------+--------------+-----+--------------------+
|                id|      category|count|                geom|
+------------------+--------------+-----+--------------------+
|617550887262617599|     education|    4|POLYGON ((2.41016...|
|617550887262617599|          park|    1|POLYGON ((2.41016...|
|617550887263141887|          park|    1|POLYGON ((2.40882...|
|617550887263141887|transportation|    1|POLYGON ((2.40882...|
|617550887263404031|       theatre|    1|POLYGON ((2.41481...|
|617550887263928319|     education|    2|POLYGON ((2.41448...|
|617550887263928319|transportation|    1|POLYGON ((2.41448...|
|617550887264190463|        bakery|    3|POLYGON ((2.40583...|
|617550887264190463|     education|    6|POLYGON ((2.40583...|
|617550887264190463|        school|    2|POLYGON ((2.40583...|
+------------------+--------------+-----+--------------------+
only showing top 10 rows



# Pivot

In [15]:
feature_df = sedona.table("catchments_count").\
    groupBy("id", "geom").pivot("category", categories).\
    agg(f.first("count")).\
    selectExpr(
        "id",
        "geom",
        "COALESCE(restaurant, 0) AS restaurant",
        "COALESCE(shopping, 0) AS shopping",
        "COALESCE(bakery, 0) AS bakery",
        "COALESCE(education, 0) AS education",
        "COALESCE(school, 0) AS school",
        "COALESCE(pharmacy, 0) AS pharmacy",
        "COALESCE(cafe, 0) AS cafe",
        "COALESCE(theatre, 0) AS theatre",
        "COALESCE(transportation, 0) AS transportation",
        "COALESCE(park, 0) AS park",
        "COALESCE(hospital, 0) AS hospital"
    )

In [16]:
feature_df.\
    withColumn("id", f.concat(f.expr("substring(id, 0, 5)"), f.lit("..."))).\
    show(5)

[Stage 12:==================================>                       (3 + 2) / 5]

+--------+--------------------+----------+--------+------+---------+------+--------+----+-------+--------------+----+--------+
|      id|                geom|restaurant|shopping|bakery|education|school|pharmacy|cafe|theatre|transportation|park|hospital|
+--------+--------------------+----------+--------+------+---------+------+--------+----+-------+--------------+----+--------+
|61755...|POLYGON ((2.46931...|         3|       0|     0|        0|     1|       1|   0|      0|             1|   0|       0|
|61755...|POLYGON ((2.28190...|         3|       1|     2|        0|     1|       4|   0|      3|             1|   5|       1|
|61755...|POLYGON ((2.25092...|        20|       6|     5|        7|     1|       8|   0|      2|             5|   7|       7|
|61755...|POLYGON ((2.22258...|        11|       5|     2|        3|     1|       3|   0|      0|             3|   1|       3|
|61755...|POLYGON ((2.26283...|         3|       0|     0|        0|     0|       0|   1|      1|             0

# Kmeans

## Create features vector

In [17]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.clustering import KMeans

assembler = VectorAssembler(
    inputCols=categories,
    outputCol="features",
)

assembled_df = assembler.transform(feature_df)

scaler = MinMaxScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scaled_data = scaler.fit(assembled_df).transform(assembled_df)

In [18]:
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, DenseVector
from pyspark.sql.types import ArrayType, DoubleType

@udf(ArrayType(DoubleType()))
def round_vector(v):
    if v is None:
        return None
    return [round(float(x), 4) for x in v]  # 4 decimal places

# Apply it
df_rounded = scaled_data.withColumn("scaled_rounded", round_vector("scaled_features"))

df_rounded.select("id", "scaled_rounded").show(5)

[Stage 30:==============================================>           (4 + 1) / 5]

+------------------+--------------------+
|                id|      scaled_rounded|
+------------------+--------------------+
|617550887464468479|[0.0097, 0.0, 0.0...|
|617550902214787071|[0.0097, 0.0059, ...|
|617550902723346431|[0.0647, 0.0353, ...|
|617550902737502207|[0.0356, 0.0294, ...|
|617550902751395839|[0.0097, 0.0, 0.0...|
+------------------+--------------------+
only showing top 5 rows



In [19]:
scaled_data.printSchema()

root
 |-- id: long (nullable = true)
 |-- geom: geometry (nullable = true)
 |-- restaurant: long (nullable = false)
 |-- shopping: long (nullable = false)
 |-- bakery: long (nullable = false)
 |-- education: long (nullable = false)
 |-- school: long (nullable = false)
 |-- pharmacy: long (nullable = false)
 |-- cafe: long (nullable = false)
 |-- theatre: long (nullable = false)
 |-- transportation: long (nullable = false)
 |-- park: long (nullable = false)
 |-- hospital: long (nullable = false)
 |-- features: vector (nullable = true)
 |-- scaled_features: vector (nullable = true)



## Train and predict

In [20]:
# Train KMeans model
kmeans = KMeans(k=10, seed=42, featuresCol="scaled_features")
model = kmeans.fit(scaled_data)

# Make predictions
predictions = model.transform(scaled_data)
predictions.select("id", "scaled_features", "prediction").show(5)

# Show cluster centers
print("Cluster Centers:")
for center in model.clusterCenters():
    print(center)

[Stage 169:==================================>                      (3 + 2) / 5]

+------------------+--------------------+----------+
|                id|     scaled_features|prediction|
+------------------+--------------------+----------+
|617550887464468479|(11,[0,4,5,8],[0....|         1|
|617550902214787071|[0.00970873786407...|         1|
|617550902723346431|[0.06472491909385...|         6|
|617550902737502207|[0.03559870550161...|         1|
|617550902751395839|(11,[0,6,7,10],[0...|         1|
+------------------+--------------------+----------+
only showing top 5 rows

Cluster Centers:
[0.53393228 0.49297629 0.6984642  0.37400208 0.42602776 0.45464983
 0.58841386 0.46291619 0.24168631 0.28913572 0.44055584]
[0.01926981 0.01097443 0.02353006 0.01472279 0.02165744 0.02846836
 0.00852828 0.01462416 0.02052343 0.0274118  0.05449937]
[0.32100651 0.25432915 0.42875753 0.32179253 0.42085551 0.45289542
 0.23405066 0.17078652 0.24699389 0.34465639 0.2785742 ]
[0.49274455 0.5743833  0.36091632 0.71005251 0.69496321 0.59181141
 0.42536905 0.26054591 0.75721562 0.435858

In [22]:
from sedona.spark import SedonaKepler
import json

with open("kmeans_viz_config.json") as f:
    config_kmeans = json.load(f)


In [ ]:
SedonaKepler.create_map(predictions, "predictions", config=config_kmeans)